# Acc-3: 결과 집계 (mean±std)

`outputs/metrics/ft_*.json`을 읽어 `B_결과/`에 메인 표를 쓴다.


In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd

ON_KAGGLE = Path("/kaggle/working").exists()
CWD = Path.cwd().resolve()
PROJ_ROOT = None
for c in [CWD, CWD.parent, CWD.parent.parent]:
    if (c / "outputs" / "metrics").exists() or (c / "A_프로토콜").exists():
        PROJ_ROOT = c
        break
if PROJ_ROOT is None:
    PROJ_ROOT = CWD

MET = PROJ_ROOT / "outputs" / "metrics"
RES = PROJ_ROOT / "B_결과"
RES.mkdir(parents=True, exist_ok=True)

rows = []
for p in sorted(MET.glob("ft_*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    if "history" in d:
        d = {k: v for k, v in d.items() if k != "history"}
    d["file"] = p.name
    rows.append(d)

if not rows:
    print("No ft_*.json yet under", MET)
else:
    df = pd.json_normalize(rows)
    df.to_csv(RES / "00_runs_raw.csv", index=False)
    keys = ["tag", "label_frac"]
    metrics = [
        "crc_acc_best",
        "crc_macro_f1_best",
        "crc_acc_last",
        "best_epoch",
        "best_holdout_acc",
    ]
    # corruption columns if present
    for c in ["color_shift", "blur", "noise", "gray"]:
        col = f"corruption_best.{c}"
        if col in df.columns:
            metrics.append(col)

    lines = ["# 메인 표 (best-val → CRC-VAL)", "", f"runs: {len(df)}", ""]
    for lf, g1 in df.groupby("label_frac"):
        lines.append(f"## label_frac={lf} (분모=FT train full)")
        lines.append("")
        lines.append("| tag | n | CRC Acc mean±std | macro-F1 mean±std | best_epoch mean |")
        lines.append("| --- | ---: | --- | --- | ---: |")
        for tag, g in g1.groupby("tag"):
            n = len(g)
            acc = g["crc_acc_best"].astype(float)
            f1 = g["crc_macro_f1_best"].astype(float)
            be = g["best_epoch"].astype(float)
            lines.append(
                f"| {tag} | {n} | {acc.mean():.4f}±{acc.std(ddof=0):.4f} | {f1.mean():.4f}±{f1.std(ddof=0):.4f} | {be.mean():.1f} |"
            )
        lines.append("")
    out_md = RES / "01_main_table.md"
    out_md.write_text("\n".join(lines), encoding="utf-8")
    print(out_md.read_text(encoding="utf-8"))
